# Graph Generation

Given node embeddings in the Poincaré disk, this notebook explores strategies for generating graphs — the *generative direction* of the Poincaré Maps model.

We discuss three approaches:
1. **Threshold on hyperbolic distance** (hyperbolic RGG)
2. **Fermi-Dirac model** (standard in the hyperbolic networks literature)
3. **Greedy rounding** of the forest matrix inverse

> Note: the `generation` module is a stub in the current release. This notebook demonstrates the approaches using the low-level functions directly.

In [ ]:
import networkx as nx
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hypegrl.embedders.poincare_maps import (
    PoincareMapsEmbedder, forest_matrix, soft_decoder
)
from hypegrl.manifolds.poincare import POINCARE_BALL
from hypegrl.visualization.disk import plot_poincare_graph

## Fit embeddings on a reference graph

In [ ]:
G_ref = nx.balanced_tree(r=2, h=3)
embedder = PoincareMapsEmbedder(
    d=2, n_steps=300, log_every=0,
    regularize_a=0.01, random_state=0,
)
embedder.fit(G_ref)
X = embedder.embeddings()
X_t = torch.tensor(X, dtype=torch.float64)

## Approach 1: Threshold on hyperbolic distance

Declare an edge $(i,j)$ if $d_H(x_i, x_j) < \tau$. The threshold $\tau$ is calibrated to match the target number of edges.

In [ ]:
D = POINCARE_BALL.dist(X_t.unsqueeze(1), X_t.unsqueeze(0)).detach().numpy()
np.fill_diagonal(D, np.inf)

target_edges = G_ref.number_of_edges()
# Binary search for threshold
flat = D[np.triu_indices_from(D, k=1)]
tau = np.sort(flat)[target_edges - 1]

N = G_ref.number_of_nodes()
G_rgg = nx.Graph()
G_rgg.add_nodes_from(range(N))
for i in range(N):
    for j in range(i+1, N):
        if D[i, j] < tau:
            G_rgg.add_edge(i, j)

print(f'Reference edges: {G_ref.number_of_edges()}')
print(f'RGG edges: {G_rgg.number_of_edges()}')
print(f'Edge overlap: {len(set(map(frozenset, G_ref.edges())) & set(map(frozenset, G_rgg.edges())))}')

## Approach 2: Fermi-Dirac model

$$P_{ij} = \frac{1}{e^{(d_H(x_i,x_j) - R)/T} + 1}$$

where $R$ controls sparsity and $T$ controls sharpness. Sample $a_{ij} \sim \text{Bernoulli}(P_{ij})$.

In [ ]:
R = np.median(flat)  # radius ~ median pairwise distance
T = 0.1              # low temperature = sharp threshold

P = 1.0 / (np.exp((D - R) / T) + 1.0)
np.fill_diagonal(P, 0.0)

rng = np.random.default_rng(0)
A_fd = (rng.random((N, N)) < P).astype(float)
A_fd = np.triu(A_fd, 1)
A_fd = A_fd + A_fd.T  # symmetrise

G_fd = nx.from_numpy_array(A_fd)
print(f'Fermi-Dirac edges: {G_fd.number_of_edges()}')

## Approach 3: Greedy rounding of the forest matrix

Compute $\hat{Q}$ from embeddings via the decoder, invert to get $\hat{L} = \hat{Q}^{-1} - I$, read off edge weights as $w_{ij} = \max(0, -\hat{L}_{ij})$, then threshold.

In [ ]:
A_hat = soft_decoder(X_t, gamma=1.0).detach().numpy()
# Q_hat ~ A_hat here; invert to get L_hat
Q_hat = A_hat
L_hat = np.linalg.inv(Q_hat) - np.eye(N)

# Off-diagonal weights: w_ij = max(0, -L_hat_ij)
W = np.maximum(0.0, -L_hat)
np.fill_diagonal(W, 0.0)
W = (W + W.T) / 2  # symmetrise numerical noise

# Threshold at median nonzero weight
nonzero = W[W > 0]
if len(nonzero) > 0:
    tau_w = np.percentile(nonzero, 60)
    G_gr = nx.from_numpy_array((W > tau_w).astype(float))
    print(f'Greedy rounding edges: {G_gr.number_of_edges()}')
else:
    print('No positive weights found — L_hat does not have Laplacian structure')

## Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, G_gen, title in zip(
    axes,
    [G_rgg, G_fd, G_ref],
    ['Hyperbolic RGG', 'Fermi-Dirac', 'Reference (fitted)'],
):
    plot_poincare_graph(G_gen, X, title=title, ax=ax)
fig.tight_layout()
fig.savefig('graph_generation.png', dpi=150, bbox_inches='tight')